# CSE 25 - Introduction to Artificial Intelligence
## Week 6 Thursday: Probability foundations for language models

**Learning Objectives:**

- Define and use basic probability concepts (sample space, events, random variables)
- Explain and apply discrete probability distributions (Bernoulli, categorical)
- Compute and interpret joint, marginal, and conditional probabilities
- Use the chain rule and independence in probability calculations
- Connect probability theory to the basics of language modeling

Instructions

Use your copy of this notebook on Datahub and complete it during class. Work through the cells below **in order**. You may discuss with your neighbors, but make sure you understand each step yourself.

SUBMISSION:
When finished, download the notebook to have a local copy for your records. Choose two cells where you wrote code or answers and take screenshots of them to upload to Gradescope under `In-Class – Week 6 Thursday` to receive credit. 

We explored linear models and neural networks and their use in regression and classification. For multi-class classification problems, the softmax activation function in the output neuron of a neural network converts raw scores into a probability distribution. This means assigning a number between 0 and 1 (inclusive) to each possible class, where the probabilities sum to 1, and the probability is associated with the confidence or likelihood that a class is the correct one.

A **language model** is this idea extended to words: it defines a probability distribution over possible strings.

#### Activity: Play Semantris

We will start by playing a game called [Semantris](https://research.google.com/semantris/)

Play the game and observe **how the AI decides which words are related**. (3-5 minutes)

1. Open **Arcade Mode**.
2. Play one or two games.
3. Type a word that you think is **related to the clue**.

For each clue, try different kinds of words:

- A **synonym** (same meaning)
- A word with a **similar root or prefix**
- An **antonym** (opposite meaning)
- A word that is **completely unrelated**

Observe how the system responds to each type of word.

Q. Write down what you observed while playing.

- What kinds of words tended to work best?
- Did the system ever accept a word that was **not an exact synonym**?
- Did the system ever reject a word that **you thought should have worked**?
- What happened when you entered **completely unrelated words**?

Write your observations below:

Task is to find related words, but sometimes random words still score points.
Commonly used words seem to score more highly.
Maybe there's a library of words that forms the training set.
Related meanings -- can we capture which words are similar to one another in ML models.
Sometimes clusters of words can point in a particular meaning for a word, instead of other meanings that the word can have in other contexts.
Antonyms can score highly - they're still related to the original words.
Meaning is not everything: shared prefix also scored highly.

System didn't let us enter word that seemed like maybe it would the same as the clue.

Other languages!

Semantris assigns scores to certain words. How does an AI system decide which words are similar or appropriate for a given task?

### Why do we need language models?

Many natural language processing (NLP) tasks require natural language **output**:
- Machine translation
- Speech recognition
- Natural language generation
- Spell checking

A language model defines a *probability distribution* over (natural language) strings or sentences.

These probability distributions then give a ranking of possible output strings so that we can choose the best: if the language model assigns a higher probability to one rather than the other, 
we are more likely to output the first string rather than the second.

**Natural Language Generation**

The sky is _____ 

`blue`

or 

`cup`

**Spell Check** 

`Their are two tests.`

or

`There are two tests.`

**Grammar**

`Everything has improve.`

or 

`Everything has improved.`

**Speech Recognition**

`I will be back sooninsh.`

or

`I will be bassoon dish.`

#### Key idea
**Generate language by outputting higher probability sentences according to the language model.**

*Connection to what you've already seen:* Softmax in a neural network converts raw scores into a probability distribution over classes. A language model does the same thing, where the "classes" are words. Every time a language model picks the next word, it is running a categorical distribution, exactly like the output layer of the networks from before.

### Probability fundamentals

You'll see a more thorough treatment of probability in CSE 21 and CSE 103. In this class, we'll introduce some key probabilistic ideas necessary to understand how language models work, without going into general properties in depth.

#### Probability Terminology

- **Trial (experiment)**  
  A single random process.  
  Examples: picking a shape from a bag, rolling a die, predicting the next word.

- **Outcome**
  A single possible result of one trial.
  Examples: picking one specific shape e.g., a blue square.

- **Sample space ($\Omega$)**  
  The set of all possible outcomes of a trial.  
  Examples: all shapes in the bag, all numbers on a die, all words in a vocabulary.

- **Event ($A \subseteq \Omega$)**  
  A subset of possible outcomes.  
  Examples: "drawing a blue shape", "rolling an even number", "predicting the word *the*".

- **Probability (of an event)**
  A number between $0$ and $1$ that measures how likely event $A$ is. Informally, the smaller the probability the less likely the event is, and the larger the probability the more certain the event is. If $A$ and $B$ are events that cannot both happen simultaneously (they are disjoint sets), then:
   $$
   P(A \cup B) = P(A) + P(B)
   $$

- **(Discrete) Probability distribution**
  A probability distribution is **discrete** if there is a fixed (often finite) set of possible outcomes. This means we can list all possible outcomes and assign a probability to each one.

- **Uniform distribution**
  Every outcome has the same probability. For a sample space with $M$ possible outcomes, the uniform distribution means all outcomes are equally likely, so the probability of each specific outcome is $1/M$.

- **Sample / sampling**
  The act of randomly drawing one outcome from a distribution.  Running the experiment once.
  
- **Sampling with replacement** 
  After drawing, the outcome is put back so the distribution does not change.  Each draw is independent.
  To contrast: *sampling without replacement* means after drawing, the outcome is *not* returned and the distribution changes.

- **Joint probability $P(A, B)$**
  The probability that two events both occur.
  Example: (blue AND square)

- **Conditional probability $P(A | B)$**
  The probability of event $A$ given that event $B$ is already known to be 
  true. 
    $$
    P(X | Y) = \frac{P(X, Y)} {P(Y)}
    $$ 
  Example: $P(\text{blue} | \text{square})$ means "given we drew a square, how likely is it blue?"

- **Independence**
  $A$ and $B$ are independent events if knowing $B$ gives no information about $A$. Symbolically, $P(A | B) = P(A)$. Equivalently, using the definition of conditional probability above, events $A$ and $B$ are independent means
  $$
   P(A, B) = P(A) \cdot P(B)
  $$
  Example: Color and shape would be independent when $P(\text{blue} | \text{square}) = P(\text{blue})$


#### Examples
<img src="images/box-of-shapes.png" width="500">

| Item | Count | 
|------|-------|
| Blue square | 2 | 
| Blue triangle | 2 | 
| Blue circle | 2 | 
| Red square | 2 | 
| Red triangle | 2 | 
| Red circle | 1 | 
| Yellow square | 2 | 
| Yellow triangle | 3 | 
| Yellow circle | 2 | 

Consider sampling with replacement, where each sample means pick a random shape from the box of shapes, then put it back in the bag.

#### Events

$P(\text{blue}) = \frac{6}{18} = \frac{1}{3}$

$P(\text{square}) = \frac{6}{18} = \frac{1}{3}$

$P(\text{square or triangle}) = \frac{13}{18} = \frac{6}{18} + \frac{7}{18} = 1 - \frac{5}{18}$

$P(\text{blue square or a red triangle}) = \frac{2}{18} + \frac{2}{18} = \frac{4}{18}$

#### Conditional probability and independence

$P(\text{blue} | \text{square}) = \frac{2}{6}$

$P(\text{blue square}) = \frac{2}{18}$

$P(\text{square}) = \frac{6}{18}$

Check: does $P(\text{blue} | \text{square}) = \frac{P(\text{blue square}) }{P(\text{square})}$? 

Calculate:

$LHS = \frac{2}{6} = \frac{1}{3}$

$RHS = \frac{ \frac{2}{18}}{\frac{6}{18}} = \frac{2}{6} = \frac{1}{3}$

#### Sequences

$P(\text{red circle, yellow triangle, blue square}) =\frac{1}{18} \cdot \frac{3}{18} \cdot \frac{2}{18} = \frac{6}{18^3} \approx 0.00103$

$P(\text{red triangle, yellow circle, red triangle}) = \frac{2}{18} \cdot \frac{2}{18} \cdot \frac{2}{18} = \frac{8}{18^3} \approx 0.00137$

$P(\text{red triangle, red triangle, yellow circle}) =$ Exactly like previous example!

**Notice:** The last two sequences contain the exact same shapes in a different order. Did you get the same probability for both? Why or why not? (Hint: think about sampling *with replacement*.)

In [ ]:
# Box of shapes
counts = {
    'blue_square': 2, 'blue_triangle': 2, 'blue_circle': 2,
    'red_square': 2,  'red_triangle': 2,  'red_circle': 1,
    'yellow_square': 2, 'yellow_triangle': 3, 'yellow_circle': 2,
}

total = sum(counts.values())

# Compute P(blue)

# Compute P(square)

# Compute P(blue, square)

# Compute P(blue | square)

# Compute P(square | blue)

### Language models

For language models, the sample space consists of text (words, or phrases, or sentences).

Suppose we have some text: 

"the cat sat on the mat . the cat scared the rat that was near the mat"

<img src="images/box-of-text.png" width="500">

| Token | Count | 
|------|-------|
| the | 5 | 
| cat | 2 | 
| mat | 2 | 
| on | 1 | 
| sat | 1 | 
| near | 1 | 
| rat| 1 | 
| scared | 1 | 
| that | 1 | 
| was | 1 | 
| . | 1 | 

Continue sampling with replacement, where each sample means pick a random text from the box, then put it back.

$P(\text{cat}) = 2/17$

$P(.) = 1/17$

$P(\text{the}) = 5/17$

This is a *simplified language model*: the probability of a specific word coming next is the frequency of the word in the training text.

Which of the following sentences has higher probability in this model?

A: `the cat sat on the mat . the cat scared the rat that was near the mat .`

B: `on the near cat the cat mat scared that the sat mat . the rat was  the  .`

And why?

A and B have the same probability! Because they have the same count of each token.

Does this probability model capture anything about the *grammar* or *meaning* of a sentence? 

No!

### Language models: some more terminology

Definition: A *language model* over a vocabulary $V$ assigns probabilities to strings drawn from $V^*$

**The Vocabulary $V$**

The vocabulary $V$ is the set of allowed words (or tokens).

Example:

$$
V = \{\text{the}, \text{cat}, \text{mat},\text{on}, \text{sat}, \text{near}, \text{rat},
\text{scared}, \text{that}, \text{was}, \text{.}\}
$$

These are the basic building blocks.


**What Is $V^*$?**

$V^*$ is the collection of possible finite sequences of words from $V$.

If

$$
V = \{\text{the}, \text{cat}\}
$$

then $V^*$ includes:

- the  
- cat  
- the cat  
- cat the  
- the the  
- cat cat  
- the cat the  
- cat the cat  

etc.


Q: For the vocabulary $V =\{ the, cat \}$, what's the size of $V$? Is it finite or infinite?

2

Q: For the same vocabulary, what's the size of $V^*$? Is it finite or infinite?

Infinite set.


A language model assigns a probability to **every possible sentence** made from the vocabulary. Formally, it defines a function:

$$
P : V^* \rightarrow [0,1]
$$

This means:

- Every possible sentence gets a probability.
- The probabilities over all possible sentences add up to 1.

$$
\sum_{s \in V^*} P(s) = 1
$$

**Why Is This Important?**

We want to compare sentences like:

- “I agree.”
- “I completely agree.”
- “Completely I agree.”

To decide which is more likely, they must come from the **same probability distribution**.

That’s why a language model defines probabilities over *all possible strings*, not just individual words.


#### Building a Probability Model

1. Define the model.
2. Parameters are conditional probabilities for the words. Estimate parameters.

Models often make independence assumptions to reduce the number of parameters.

### Language models: removing independent words assumption

For language models with uniform probability distribution over words and where words are independent, when $s_1$ and $s_2$ are sentences of equal length,

$$P_{model}(s_1) = P_{model}(s_2)$$

We can do better!

**IDEA** Likelihood of a word changes based on the words that came before it.

Based on the definition of joint probability and conditional probability, 

$$
P(w_1, w_2, \dots, w_n)
=
P(w_1) \cdot 
P(w_2 \mid w_1) \cdot 
P(w_3 \mid w_1, w_2)
\cdots
P(w_n \mid w_1, \dots, w_{n-1})
$$

<!-- More compactly,

$$
P(w_1, \dots, w_n)
=
\prod_{k=1}^{n}
P(w_k \mid w_{1:k-1})
$$ -->

(Most of) the terms in the product are the probability of the next word given some **history**, $P(w \mid h)$.

*Example*

Suppose the history is:

> *On summer evenings the sky looks very*

and we want the probability that the next word is *orange*:

$$
P(\text{orange} \mid \text{On summer evenings the sky looks very})
$$

A simple idea is to estimate this using counts from a large corpus.

We count:

- How often we see the full sequence  
  *On summer evenings the sky looks very orange*

- How often we see the history  
  *On summer evenings the sky looks very*

This gives the relative-frequency estimate:

$$
P(w \mid h)
=
\frac{C(h\,w)}{C(h)}
$$

In words:  "Out of all the times we saw the history $h$, how often was it followed by the word $w$?"

Q. Why is estimating  
$$
P(w_n \mid w_1, w_2, \dots, w_{n-1})
$$
for long histories unrealistic in practice?

- Sparsity of specific sentences in training data
- Exponential costs associated with training to estimate these conditional probabilities and also would like to very large models to store these parameters.

If we had a large enough text sample (also known as corpus), we could compute all these counts. 

However, even the entire web is not large enough to give reliable counts for long histories. Language is creative. New sentences are invented all the time. Most long word sequences will appear rarely, or never, in our data. We cannot expect to see every possible long history $h$ in our training data. If a sequence never appears, our estimate becomes zero. That would mean assigning probability zero to perfectly reasonable sentences.

Today we saw that a language model is a probability distribution over all possible strings. Computing $P(w_n |w_1, \ldots, w_{n-1})$ directly from counts fails for long histories because most long sequences never appear in training data.

Next time we'll see compromise approaches that keep some, but not all, of the history. These **$n$-gram** models (bigrams, trigrams) approximate the full history with just the last 1-2 words, making estimation more tractable.